# Lecture 7 — Classroom Exercise (Real API Data + Plotly)

**Goal:** Practice building a **decision-ready** interactive visualization.

You will:
1. Create folders (`data/raw`, `data/clean`, `output/figures`, …)
2. Pull **real data** from the BLS API (CPI and Unemployment)
3. Save raw + clean files
4. Build Plotly charts with:
   - labels + units
   - benchmark line (2%)
   - at least one annotation
5. Write short interpretations (you will be graded on clarity, not just code)

> If Plotly is not installed, run the install cell and restart the kernel.


In [2]:
# If Plotly is missing in this environment, install it here.
# IMPORTANT: After installation, restart the kernel (Kernel → Restart) and re-run imports.

import sys, importlib.util

if importlib.util.find_spec("plotly") is None:
    print("Plotly not found. Installing...")
    !{sys.executable} -m pip install -q plotly
else:
    print("Plotly is already installed.")


Plotly is already installed.


In [4]:
from pathlib import Path
import json
import requests
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from datetime import datetime

# Make Plotly look nice in notebooks
import plotly.io as pio
pio.renderers.default = "notebook_connected"


In [6]:
# -------------------------
# (1) Create project folders
# -------------------------
PROJECT_ROOT = Path.cwd()

DIRS = {
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_clean": PROJECT_ROOT / "data" / "clean",
    "output": PROJECT_ROOT / "output",
    "figures": PROJECT_ROOT / "output" / "figures",
    "tables": PROJECT_ROOT / "output" / "tables",
    "scripts": PROJECT_ROOT / "scripts",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("Folders ready:")
for k, p in DIRS.items():
    print(f"  {k:10s} -> {p}")


Folders ready:
  data_raw   -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/raw
  data_clean -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/clean
  output     -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/output
  figures    -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/output/figures
  tables     -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/output/tables
  scripts    -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/scripts


## Task 0 — Helper functions (provided)

In [8]:
def save_json(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2))
    print(f"Saved raw JSON: {path}")

def bls_timeseries(series_ids, start_year=2015, end_year=2025, api_key=None):
    """Pull time series from the BLS Public Data API v2."""
    url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
    payload = {
        "seriesid": series_ids,
        "startyear": str(start_year),
        "endyear": str(end_year),
    }
    if api_key:
        payload["registrationkey"] = api_key

    r = requests.post(url, json=payload, timeout=60)
    r.raise_for_status()
    out = r.json()
    # Basic sanity check
    if out.get("status") != "REQUEST_SUCCEEDED":
        raise RuntimeError(f"BLS API request failed: {out}")
    return out

def bls_to_tidy_df(bls_json):
    """Convert BLS JSON to tidy monthly data."""
    rows = []
    for s in bls_json["Results"]["series"]:
        sid = s["seriesID"]
        for item in s["data"]:
            # Monthly periods are M01..M12 (M13 is annual avg; skip it)
            period = item.get("period", "")
            if period.startswith("M") and period != "M13":
                year = int(item["year"])
                month = int(period[1:])
                date = pd.Timestamp(year=year, month=month, day=1)
                rows.append({
                    "series_id": sid,
                    "date": date,
                    "value": float(item["value"]),
                })
    df = pd.DataFrame(rows)
    df = df.sort_values(["series_id", "date"]).reset_index(drop=True)
    return df


## Task 1 — Pull real data and save raw JSON (10 pts)

Use the BLS API to pull:
- CPI-U All items SA: `CUSR0000SA0`
- Unemployment rate SA: `LNS14000000`

Save the raw JSON into `data/raw/`.


In [10]:
# YOUR CODE HERE

SERIES = {
    "cpi_u_all_items": "CUSR0000SA0",
    "unemp_rate": "LNS14000000",
}

start_year, end_year = 2015, 2025

# 1) call the API
bls_json = bls_timeseries(list(SERIES.values()), start_year=start_year, end_year=end_year)

# 2) save raw json
raw_path = DIRS["data_raw"] / f"bls_timeseries_{start_year}_{end_year}.json"
save_json(bls_json, raw_path)

# 3) tidy df
tidy = bls_to_tidy_df(bls_json)
tidy.head()


Saved raw JSON: /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/raw/bls_timeseries_2015_2025.json


,series_id,date,value
0,CUSR0000SA0,2015-01-01,234.747
1,CUSR0000SA0,2015-02-01,235.342
2,CUSR0000SA0,2015-03-01,235.976
3,CUSR0000SA0,2015-04-01,236.222
4,CUSR0000SA0,2015-05-01,237.001


## Task 2 — Build a clean monthly dataset (15 pts)

Create `wide` with one row per month and columns:
- `cpi_u_all_items`
- `unemp_rate`
- `cpi_yoy_pct` (YoY % change)
- `cpi_mom_pct` (MoM % change)

Save the clean CSV into `data/clean/`.


In [ ]:
# YOUR CODE HERE

id_to_name = {v: k for k, v in SERIES.items()}
tidy["series_name"] = tidy["series_id"].map(id_to_name)

wide = tidy.pivot_table(index="date", columns="series_name", values="value").reset_index()
wide = wide.sort_values("date")

wide["cpi_yoy_pct"] = wide["cpi_u_all_items"].pct_change(12) * 100
wide["cpi_mom_pct"] = wide["cpi_u_all_items"].pct_change(1) * 100

clean_path = DIRS["data_clean"] / f"monthly_cpi_unemp_{start_year}_{end_year}.csv"
wide.to_csv(clean_path, index=False)
print(f"Saved clean data: {clean_path}")

wide.tail()


## Task 3 — Build a decision-ready CPI YoY chart (25 pts)

Requirements:
- Plot `cpi_yoy_pct` over time with Plotly
- Title must state the decision question (not just “CPI”)
- Axis labels include units (%)
- Add a benchmark line at 2%
- Add **one annotation** (COVID marker OR peak month)

Then: write 3–4 sentences interpreting what the chart suggests for a decision-maker.


In [ ]:
from datetime import datetime
# YOUR CODE HERE

df_plot = wide.dropna(subset=["cpi_yoy_pct"]).copy()

fig = px.line(
    df_plot, x="date", y="cpi_yoy_pct",
    title="Is inflation near a 2% benchmark? CPI YoY inflation over time"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Year-over-Year change (%)",
    hovermode="x unified"
)

fig.add_hline(y=2.0, line_dash="dash", annotation_text="Benchmark: 2%")

# Choose ONE:
# Event marker (robust to pandas/plotly datetime issues)
x_event = datetime(2020, 3, 1)
fig.add_shape(
    type="line",
    x0=x_event, x1=x_event,
    y0=0, y1=1,
    xref="x", yref="paper",
    line=dict(dash="dot")
)
fig.add_annotation(
    x=x_event, y=1, xref="x", yref="paper",
    text="COVID shock",
    showarrow=False,
    yanchor="bottom"
)
# OR annotate the peak
# peak_row = df_plot.loc[df_plot['cpi_yoy_pct'].idxmax()]
# fig.add_annotation(x=peak_row["date"], y=peak_row["cpi_yoy_pct"],
#                    text=f"Peak: {peak_row['cpi_yoy_pct']:.1f}%",
#                    showarrow=True, arrowhead=2)

fig.update_traces(hovertemplate="<b>%{x|%Y-%m}</b><br>%{y:.2f}%")
fig.show()


### Interpretation (write below) — 10 pts

- What is the “main story” of inflation over the sample?
- What does the 2% benchmark add?
- If you were advising someone, what would you watch next month?


## Task 4 — Axis choice debate (20 pts)

Make **two** CPI charts:

- **Version A:** y-axis forced to include 0  
- **Version B:** y-axis zoomed (e.g., -1 to 10)

Then answer:
1) Which is better for a general audience? Why?  
2) Which is better for technical monitoring? Why?  
3) When does zooming become misleading?

*(You are being graded on reasoning.)*


In [ ]:
# YOUR CODE HERE

# Version A: include 0
figA = px.line(df_plot, x="date", y="cpi_yoy_pct", title="Version A: Include 0 baseline")
figA.update_yaxes(range=[0, max(10, float(df_plot["cpi_yoy_pct"].max()) + 1)])
figA.update_layout(xaxis_title="Date", yaxis_title="YoY change (%)", hovermode="x unified")
figA.show()

# Version B: zoom
figB = px.line(df_plot, x="date", y="cpi_yoy_pct", title="Version B: Zoomed axis (monitoring)")
figB.update_yaxes(range=[-1, 10])
figB.update_layout(xaxis_title="Date", yaxis_title="YoY change (%)", hovermode="x unified")
figB.show()


## Task 5 — Inflation vs Unemployment (bonus +10 pts)

Create a two-axis Plotly chart:
- CPI YoY on left axis
- Unemployment rate on right axis

Then write 2 sentences:
- One pattern you see
- One thing you **cannot** conclude (causality)


In [ ]:
# BONUS: YOUR CODE HERE

df2 = wide.dropna(subset=["cpi_yoy_pct","unemp_rate"]).copy()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df2["date"], y=df2["cpi_yoy_pct"], mode="lines", name="CPI YoY (%)"))
fig2.add_trace(go.Scatter(x=df2["date"], y=df2["unemp_rate"], mode="lines", name="Unemployment (%)", yaxis="y2"))

fig2.update_layout(
    title="Inflation vs Unemployment (Two-axis view)",
    xaxis=dict(title="Date"),
    yaxis=dict(title="CPI YoY (%)"),
    yaxis2=dict(title="Unemployment (%)", overlaying="y", side="right"),
    hovermode="x unified",
)

fig2.show()


## Submission checklist

- Raw JSON saved to `data/raw/`
- Clean CSV saved to `data/clean/`
- Task 3 chart includes benchmark + annotation + good labels
- Interpretation paragraphs completed (Tasks 3 and 4)
